In [1]:
import numpy as np
import pandas as pd

In [2]:

### Inputs for the beam parameters found in the tables in the AISC manual.
### I wanted to try to import the tables to python, but I couldn't find a good way to import them from a pdf.

# Loads on the beam
# The live load is an active load on the beam
# The dead load is the permanent load on the beam from factors like its own weight and the weight of other members it supports
dead = float(input('Member dead load:'))
live = float(input('Member live load:'))

# The type of member we are calculating for (e.g. W8x24) and is the member we will look for in the tables.
w = input('W member name:')

# The minimum yield and tensile strengths of the material of the beam given in table 2-4 of the AISC manual
fy = float(input('Member yield stress:'))
fu = float(input('Member tensile stress:'))

# These are the dimensions of the W member given in table 1-1:
ag = float(input('Member gross area:'))
bf = float(input('Member flange width:'))
tf = float(input('Member flange thickness:'))
d = float(input('Member depth:'))
ry = float(input('Member y centroid:'))
ybar = float(input('Xbar of the WT member associated with the inputed W member:'))

# These are the yield and rupture strengths for LFRD and ASD of the member given in table 5-1
Ylfrd = float(input('Member LFRD yield strength:'))
Yasd = float(input('Member ASD yield stregnth:'))
Rlfrd = float(input('Member LFRD rupture stregnth:'))
Rasd = float(input('Member ASD rupture stregnth:'))

# Number of bolts in the member
bolts = float(input('Number of bolts in member:'))
# Diameter of the bolt holes, generally 1/16th inch greater than the size of the bolt
dh = float(input('Diameter of bolt holes:'))
# Distance between bolt holes
spacing = float(input('Spacing between bolts:'))


Member dead load: 0
Member live load: 0
W member name: 0
Member yield stress: 0
Member tensile stress: 0
Member gross area: 0
Member flange width: 0
Member flange thickness: 0
Member depth: 0
Member y centroid: 0
Member LFRD yield strength: 0
Member ASD yield stregnth: 0
Member LFRD rupture stregnth: 0
Member ASD rupture stregnth: 0
Number of bolts in member: 0
Diameter of bolt holes: .75
Spacing between bolts: 0
Xbar of the WT member associated with the inputed W member: 0


In [5]:
#calculates the required strength of the member based on LFRD standard
def required_strength_LFRD(live, dead):
    load = 1.2 * dead + 1.6 * live
    return load

#calculates the required strength based on ASD standard
def required_strength_ASD(live, dead):
    load = dead + live
    return load

#calculates effective net area
def ag_ae_ratio(bf,tf,Ag,ybar,spacing,bolts,d,dh):
    #first we need to calculate shear lag factor U
    #the minimum value that U can be is defined as:
    Umin = 2 * bf * tf / Ag
    #there are two possible cases for U
    # U can be defined as:
    U1 = 1 - ybar / (spacing * bolts)
    # or it can be the following values depending on the dimensions of the beam
    if bf < 2/3 * d:
        U2 = 0.85
    else:
        U2 = 0.9
    # we take the larger of the U's
    U = max(Umin, U1, U2)
    #Net area is found as an intermediate step
    # its basically the area of the cross-section minus the area of bolt holes
    An = Ag - 4 * (dh + 1/16) * tf
    #we use An and U to calculate Ae
    Ae = An*U
    return Ae

In [8]:
#uses the strength functions to define variables
LFRD_strength = required_strength_LFRD(live, dead)
ASD_strength = required_strength_ASD(live, dead)

#checks the required yield strength against the actual strength of the member
if LFRD_strength > Ylfrd:
    print("This member's LFRD yield strength is insufficent.")
elif ASD_strength > Yasd:
    print("This member's ASD yield strength is insufficent.")
else:
    print("This member's tensile yield stregnth is sufficent.")
    

This memeber's tensile yield stregnth is ok


In [ ]:
# uses the Ae function
Ae = ag_ae_ratio(bf,tf,Ag,ybar,spacing,bolts,d,dh)
# calculates the ratio between Ae and Ag
ratio = Ae/Ag

#if the ratio Ae/Ag is greater than .75 the values of rupture strength from table 5-1 are used
if ratio >= 0.75:
    #this compares the rupture strength of the beam to the load
    if LFRD_strength > Rlfrd:
        print("This member's LFRD rupture strength is insufficent.")
    elif ASD_strength > Rasd:
        print("This member's ASD rupture strength is insufficent.")
    else:
        print("This member's rupture strength is sufficent.")
#if the ratio of Ae/Ag is less than .75 new values of rupture strength are calculated
else:
    #the same as the comparison above, but it uses new calculated values of rupture strength
    if LFRD_strength > Fu * Ae * 0.75:
        print("This member's LFRD rupture strength is insufficent.")
    elif ASD_strength > Fu * Ae / 2:
        print("This member's ASD rupture strength is insufficent.")
    else:
        print("This member's rupture strength is sufficent.")